# Online Learning: Predictive Maintenance dengan SGDClassifier

**Konsep**: Model belajar secara bertahap (incremental) setiap kali ada data baru yang datang, tanpa perlu retrain dari nol.

**Algoritma**: `SGDClassifier` (Stochastic Gradient Descent) - mendukung `partial_fit()` untuk update model secara incremental.

**Workflow**:
1. Load data awal (batch 1) → train model pertama dengan `partial_fit`
2. Load data baru dari `NEW_DATA` → update model dengan `partial_fit`
3. Deteksi **concept drift** (perubahan pola data) dengan monitoring metrics
4. Evaluasi performa model setelah update
5. Logging ke MLflow

**Analogi**: Seperti teknisi yang belajar dari setiap mesin yang diperbaiki. Setiap kasus baru, pengetahuannya langsung di-update.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, average_precision_score,
    roc_auc_score, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------- Configuration ----------------
DATA_SOURCE = os.getenv("DATA_SOURCE", "csv")
DATA_PATH = os.getenv("DATA_PATH", "../data/ai4i2020.csv")
NEW_DATA = os.getenv("NEW_DATA", "../data/new_machine_data.csv")  # Data baru untuk incremental update

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "online-learning-machine-failure")
REGISTERED_MODEL_NAME = os.getenv("MLFLOW_MODEL_NAME", "machine_failure_sgd_online")
MODEL_ALIAS = os.getenv("MLFLOW_MODEL_ALIAS", "champion")

# Drift detection threshold
DRIFT_THRESHOLD_F1 = 0.10  # Jika F1 drop > 10%, flag sebagai drift
DRIFT_THRESHOLD_KS = 0.05  # KS statistic threshold untuk distribusi fitur

print("✅ Configuration loaded")
print(f"Initial data: {DATA_PATH}")
print(f"New data: {NEW_DATA}")

## 1. Load Initial Data (Batch 1)

Data awal untuk training model pertama.

In [ ]:
COLUMNS = ["udi", "product_id", "type", "air_temp_k", "process_temp_k",
           "rpm", "torque_nm", "tool_wear_min", "machine_failure",
           "twf", "hdf", "pwf", "osf", "rnf"]
INT_COLS = ["udi", "rpm", "tool_wear_min", "machine_failure", "twf", "hdf", "pwf", "osf", "rnf"]
FLOAT_COLS = ["air_temp_k", "process_temp_k", "torque_nm"]

def load_data(path):
    """Load data dari CSV dengan encoding yang benar."""
    print(f"📂 Loading data from: {path}")
    d = pd.read_csv(path, encoding="utf-8-sig")
    d.columns = COLUMNS
    d = d.astype({**{c: "int64" for c in INT_COLS}, **{c: "float64" for c in FLOAT_COLS}})
    print(f"   Shape: {d.shape}")
    print(f"   Failure rate: {d['machine_failure'].mean():.2%}")
    return d

# Load initial data
df_initial = load_data(DATA_PATH)
print(f"\n✅ Initial data loaded: {len(df_initial)} rows")

## 2. Feature Engineering

Reuse feature engineering dari notebook asli (XGBoost).

In [ ]:
TYPE_MAP = {"L": 0, "M": 1, "H": 2}

def make_features(d: pd.DataFrame) -> pd.DataFrame:
    """Create derived features based on physical rules."""
    out = pd.DataFrame(index=d.index)
    out["type_enc"] = d["type"].map(TYPE_MAP)
    out["air_temp_k"] = d["air_temp_k"]
    out["process_temp_k"] = d["process_temp_k"]
    out["rpm"] = d["rpm"]
    out["torque_nm"] = d["torque_nm"]
    out["tool_wear_min"] = d["tool_wear_min"]
    out["temp_diff_k"] = d["process_temp_k"] - d["air_temp_k"]
    out["power_w"] = d["torque_nm"] * d["rpm"] * 2 * np.pi / 60
    out["wear_torque"] = d["tool_wear_min"] * d["torque_nm"]
    return out

# Apply feature engineering
X_initial = make_features(df_initial)
y_initial = df_initial["machine_failure"]
FEATURES = list(X_initial.columns)

print(f"✅ Features created: {len(FEATURES)} features")
print(f"   Features: {FEATURES}")
X_initial.head()

## 3. Split Initial Data & Initialize Model

Split data awal menjadi train/test. Train model pertama dengan `partial_fit`.

In [ ]:
# Split initial data
X_train_init, X_test_init, y_train_init, y_test_init = train_test_split(
    X_initial, y_initial, test_size=0.20, stratify=y_initial, random_state=RANDOM_STATE
)

print(f"Initial train: {len(X_train_init)} rows, failure rate: {y_train_init.mean():.2%}")
print(f"Initial test:  {len(X_test_init)} rows, failure rate: {y_test_init.mean():.2%}")

# Initialize scaler dan model
scaler = StandardScaler()
model = SGDClassifier(
    loss="log_loss",  # Logistic regression
    penalty="l2",
    alpha=0.0001,
    learning_rate="adaptive",
    eta0=0.01,
    random_state=RANDOM_STATE,
    class_weight="balanced"  # Handle imbalanced data
)

# Fit scaler pada data train awal
X_train_scaled = scaler.fit_transform(X_train_init)
X_test_scaled = scaler.transform(X_test_init)

print("\n✅ Model initialized: SGDClassifier with StandardScaler")

## 4. Phase 1: Initial Training dengan `partial_fit`

Latih model pertama dengan data awal. Karena ini pertama kali, kita perlu specify `classes`.

In [ ]:
print("🚀 Phase 1: Initial Training")
print("-" * 50)

# First partial_fit: harus specify classes
classes = np.array([0, 1])
model.partial_fit(X_train_scaled, y_train_init, classes=classes)

# Evaluate pada test set awal
y_pred_init = model.predict(X_test_scaled)
y_proba_init = model.predict_proba(X_test_scaled)[:, 1]

f1_init = f1_score(y_test_init, y_pred_init)
precision_init = precision_score(y_test_init, y_pred_init)
recall_init = recall_score(y_test_init, y_pred_init)
pr_auc_init = average_precision_score(y_test_init, y_proba_init)

print(f"✅ Initial model trained on {len(X_train_init)} samples")
print(f"\n📊 Initial Performance (Test Set):")
print(f"   F1-Score:  {f1_init:.4f}")
print(f"   Precision: {precision_init:.4f}")
print(f"   Recall:    {recall_init:.4f}")
print(f"   PR-AUC:    {pr_auc_init:.4f}")

# Store metrics untuk tracking
metrics_history = [{
    "phase": "initial",
    "n_samples": len(X_train_init),
    "f1": f1_init,
    "precision": precision_init,
    "recall": recall_init,
    "pr_auc": pr_auc_init,
    "timestamp": datetime.now().isoformat()
}]

## 5. Phase 2: Load New Data & Incremental Update

Load data baru dari `NEW_DATA` dan update model dengan `partial_fit`.

In [ ]:
print("\n🔄 Phase 2: Loading New Data")
print("-" * 50)

# Check apakah file NEW_DATA ada
if not os.path.exists(NEW_DATA):
    print(f"⚠️  WARNING: {NEW_DATA} not found!")
    print("   Creating synthetic new data for demonstration...")
    
    # Buat synthetic new data (simulasi data baru dari sensor)
    # Ambil 20% dari data awal sebagai "new data"
    _, df_new = train_test_split(df_initial, test_size=0.20, random_state=RANDOM_STATE+1)
    df_new = df_new.reset_index(drop=True)
    
    # Simulasi concept drift: geser distribusi fitur sedikit
    # (misalnya mesin lebih tua, suhu rata-rata naik)
    df_new["air_temp_k"] = df_new["air_temp_k"] + np.random.normal(2, 1, len(df_new))
    df_new["process_temp_k"] = df_new["process_temp_k"] + np.random.normal(3, 1.5, len(df_new))
    df_new["tool_wear_min"] = df_new["tool_wear_min"] + np.random.randint(10, 30, len(df_new))
    
    print(f"   ✅ Synthetic new data created: {len(df_new)} rows")
    print(f"   ⚠️  Note: Simulated concept drift (temperature +2-3K, tool wear +10-30 min)")
else:
    df_new = load_data(NEW_DATA)
    print(f"✅ New data loaded: {len(df_new)} rows")

# Feature engineering untuk new data
X_new = make_features(df_new)
y_new = df_new["machine_failure"]

# Split new data menjadi train/test
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y_new, test_size=0.30, stratify=y_new, random_state=RANDOM_STATE
)

print(f"\n📊 New data split:")
print(f"   Train: {len(X_train_new)} rows, failure rate: {y_train_new.mean():.2%}")
print(f"   Test:  {len(X_test_new)} rows, failure rate: {y_test_new.mean():.2%}")

## 6. Concept Drift Detection (Sebelum Update)

Cek apakah ada perubahan distribusi antara data awal dan data baru.

**Metode**: Kolmogorov-Smirnov (KS) test untuk setiap fitur numerik.

In [ ]:
from scipy.stats import ks_2samp

print("\n🔍 Concept Drift Detection (KS Test)")
print("-" * 50)

drift_detected = False
drift_features = []

# Compare distribusi fitur antara data awal dan data baru
numeric_features = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", 
                    "tool_wear_min", "temp_diff_k", "power_w", "wear_torque"]

for feat in numeric_features:
    stat, p_value = ks_2samp(X_initial[feat], X_new[feat])
    if stat > DRIFT_THRESHOLD_KS:
        drift_detected = True
        drift_features.append(feat)
        print(f"   ⚠️  {feat:20s}: KS stat = {stat:.4f} (p-value = {p_value:.4f}) → DRIFT")
    else:
        print(f"   ✓  {feat:20s}: KS stat = {stat:.4f} (p-value = {p_value:.4f})")

if drift_detected:
    print(f"\n🚨 DRIFT DETECTED in {len(drift_features)} features: {drift_features}")
    print("   → Model perlu di-update dengan data baru")
else:
    print(f"\n✅ No significant drift detected")

## 7. Incremental Update dengan `partial_fit`

Update model dengan data baru. **Tidak perlu retrain dari nol!**

In [ ]:
print("\n🔄 Phase 3: Incremental Update dengan partial_fit")
print("-" * 50)

# Update scaler dengan data baru (incremental)
# StandardScaler tidak support partial_fit, jadi kita re-fit
# Untuk production, bisa pakai IncrementalScaler dari river library
scaler.partial_fit(X_train_new)  # Ini tidak ada di StandardScaler, jadi kita buat custom

# Workaround: combine old scaler stats dengan new data
# Untuk simplicity, kita re-fit scaler dengan combined data
X_combined_train = pd.concat([X_train_init, X_train_new], ignore_index=True)
scaler = StandardScaler()
scaler.fit(X_combined_train)

# Scale new data
X_train_new_scaled = scaler.transform(X_train_new)
X_test_new_scaled = scaler.transform(X_test_new)

# Incremental update model dengan partial_fit
# Tidak perlu specify classes lagi karena sudah di-set di initial training
model.partial_fit(X_train_new_scaled, y_train_new)

print(f"✅ Model updated with {len(X_train_new)} new samples")
print(f"   Total samples seen: {len(X_train_init) + len(X_train_new)}")

## 8. Evaluate Model Setelah Update

Cek performa model setelah di-update dengan data baru.

In [ ]:
print("\n📊 Phase 4: Evaluation After Update")
print("-" * 50)

# Evaluate pada test set baru
y_pred_new = model.predict(X_test_new_scaled)
y_proba_new = model.predict_proba(X_test_new_scaled)[:, 1]

f1_new = f1_score(y_test_new, y_pred_new)
precision_new = precision_score(y_test_new, y_pred_new)
recall_new = recall_score(y_test_new, y_pred_new)
pr_auc_new = average_precision_score(y_test_new, y_proba_new)

print(f"✅ Updated model performance (New Test Set):")
print(f"   F1-Score:  {f1_new:.4f}")
print(f"   Precision: {precision_new:.4f}")
print(f"   Recall:    {recall_new:.4f}")
print(f"   PR-AUC:    {pr_auc_new:.4f}")

# Compare dengan initial performance
print(f"\n📈 Performance Comparison:")
print(f"   {'Metric':<12} {'Initial':<12} {'After Update':<15} {'Change':<10}")
print(f"   {'-'*49}")
print(f"   {'F1':<12} {f1_init:<12.4f} {f1_new:<15.4f} {f1_new-f1_init:+.4f}")
print(f"   {'Precision':<12} {precision_init:<12.4f} {precision_new:<15.4f} {precision_new-precision_init:+.4f}")
print(f"   {'Recall':<12} {recall_init:<12.4f} {recall_new:<15.4f} {recall_new-recall_init:+.4f}")
print(f"   {'PR-AUC':<12} {pr_auc_init:<12.4f} {pr_auc_new:<15.4f} {pr_auc_new-pr_auc_init:+.4f}")

# Check apakah ada performance drop (indikasi drift yang tidak ter-handle)
f1_drop = f1_init - f1_new
if f1_drop > DRIFT_THRESHOLD_F1:
    print(f"\n⚠️  WARNING: F1 dropped by {f1_drop:.4f} (> {DRIFT_THRESHOLD_F1})")
    print("   → Model mungkin perlu retrain penuh atau tuning hyperparameter")
else:
    print(f"\n✅ Performance maintained or improved (F1 change: {f1_new-f1_init:+.4f})")

# Store metrics
metrics_history.append({
    "phase": "after_update",
    "n_samples": len(X_train_init) + len(X_train_new),
    "f1": f1_new,
    "precision": precision_new,
    "recall": recall_new,
    "pr_auc": pr_auc_new,
    "timestamp": datetime.now().isoformat()
})

## 9. Visualisasi: Performance Over Time

Plot bagaimana performa model berubah seiring bertambahnya data.

In [ ]:
print("\n📈 Visualizing Performance Evolution")
print("-" * 50)

metrics_df = pd.DataFrame(metrics_history)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# F1-Score
axes[0, 0].plot(range(len(metrics_df)), metrics_df["f1"], marker="o", linewidth=2, markersize=8)
axes[0, 0].set_xlabel("Phase")
axes[0, 0].set_ylabel("F1-Score")
axes[0, 0].set_title("F1-Score Evolution")
axes[0, 0].set_xticks(range(len(metrics_df)))
axes[0, 0].set_xticklabels(metrics_df["phase"], rotation=45)
axes[0, 0].grid(alpha=0.3)
axes[0, 0].axhline(y=metrics_df["f1"].mean(), color="r", linestyle="--", label="Mean")
axes[0, 0].legend()

# Precision & Recall
axes[0, 1].plot(range(len(metrics_df)), metrics_df["precision"], marker="s", label="Precision", linewidth=2)
axes[0, 1].plot(range(len(metrics_df)), metrics_df["recall"], marker="^", label="Recall", linewidth=2)
axes[0, 1].set_xlabel("Phase")
axes[0, 1].set_ylabel("Score")
axes[0, 1].set_title("Precision & Recall Evolution")
axes[0, 1].set_xticks(range(len(metrics_df)))
axes[0, 1].set_xticklabels(metrics_df["phase"], rotation=45)
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# PR-AUC
axes[1, 0].plot(range(len(metrics_df)), metrics_df["pr_auc"], marker="D", color="green", linewidth=2, markersize=8)
axes[1, 0].set_xlabel("Phase")
axes[1, 0].set_ylabel("PR-AUC")
axes[1, 0].set_title("PR-AUC Evolution")
axes[1, 0].set_xticks(range(len(metrics_df)))
axes[1, 0].set_xticklabels(metrics_df["phase"], rotation=45)
axes[1, 0].grid(alpha=0.3)

# Number of samples
axes[1, 1].bar(range(len(metrics_df)), metrics_df["n_samples"], color="skyblue", edgecolor="black")
axes[1, 1].set_xlabel("Phase")
axes[1, 1].set_ylabel("Total Samples Seen")
axes[1, 1].set_title("Cumulative Training Data")
axes[1, 1].set_xticks(range(len(metrics_df)))
axes[1, 1].set_xticklabels(metrics_df["phase"], rotation=45)
axes[1, 1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("✅ Visualization complete")

## 10. Confusion Matrix & Classification Report

In [ ]:
print("\n📋 Final Classification Report (New Test Set)")
print("-" * 50)
print(classification_report(y_test_new, y_pred_new, target_names=["Normal", "Failure"]))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test_new, y_pred_new),
                       display_labels=["Normal", "Failure"]).plot(cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix (After Update)")
plt.tight_layout()
plt.show()

## 11. Logging ke MLflow

Log model, metrics, dan metadata ke MLflow untuk tracking dan deployment.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
client = MlflowClient()

print("\n📝 Logging to MLflow")
print("-" * 50)

with mlflow.start_run(run_name=f"sgd_online_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
    # Log parameters
    mlflow.set_tags({
        "model_type": "SGDClassifier",
        "learning_type": "online_incremental",
        "data_source": DATA_SOURCE,
        "initial_data": DATA_PATH,
        "new_data": NEW_DATA,
        "drift_detected": str(drift_detected),
        "drift_features": json.dumps(drift_features)
    })
    
    mlflow.log_params({
        "loss": model.loss,
        "penalty": model.penalty,
        "alpha": model.alpha,
        "learning_rate": model.learning_rate,
        "eta0": model.eta0,
        "class_weight": "balanced",
        "n_features": len(FEATURES),
        "initial_train_samples": len(X_train_init),
        "new_train_samples": len(X_train_new),
        "total_samples": len(X_train_init) + len(X_train_new)
    })
    
    # Log metrics
    mlflow.log_metrics({
        "initial_f1": f1_init,
        "initial_precision": precision_init,
        "initial_recall": recall_init,
        "initial_pr_auc": pr_auc_init,
        "final_f1": f1_new,
        "final_precision": precision_new,
        "final_recall": recall_new,
        "final_pr_auc": pr_auc_new,
        "f1_change": f1_new - f1_init,
        "n_drift_features": len(drift_features)
    })
    
    # Log metrics history
    mlflow.log_dict({"metrics_history": metrics_history}, "metrics_history.json")
    
    # Log model metadata
    meta = {
        "features": FEATURES,
        "type_map": TYPE_MAP,
        "classes": [0, 1],
        "model_type": "SGDClassifier",
        "learning_type": "online_incremental"
    }
    mlflow.log_dict(meta, "meta/model_meta.json")
    
    # Log model
    sig_X = X_test_new_scaled[:5]
    signature = infer_signature(sig_X, model.predict_proba(sig_X)[:, 1])
    
    model_info = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=sig_X
    )
    
    print(f"✅ Run ID: {run.info.run_id}")
    print(f"   Model URI: {model_info.model_uri}")
    
    # Register model
    mv = mlflow.register_model(model_info.model_uri, REGISTERED_MODEL_NAME)
    
    # Set tags on model version
    tags = {
        "features": json.dumps(FEATURES),
        "type_map": json.dumps(TYPE_MAP),
        "model_type": "SGDClassifier",
        "learning_type": "online_incremental",
        "total_samples": str(len(X_train_init) + len(X_train_new)),
        "final_f1": f"{f1_new:.4f}",
        "final_pr_auc": f"{pr_auc_new:.4f}",
        "drift_detected": str(drift_detected),
        "run_id": run.info.run_id
    }
    
    for k, v in tags.items():
        client.set_model_version_tag(REGISTERED_MODEL_NAME, mv.version, k, v)
    
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS, mv.version)
    
    print(f"\n✅ Model registered: {REGISTERED_MODEL_NAME}@{MODEL_ALIAS} (version {mv.version})")

## 12. Inference Test

Test model yang sudah di-update dengan data baru.

In [ ]:
print("\n🔮 Inference Test")
print("-" * 50)

# Load model dari registry
model_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
loaded_model = mlflow.sklearn.load_model(model_uri)
mv_loaded = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
feats = json.loads(mv_loaded.tags["features"])

# Test dengan sample data
test_sample = {
    "type": "L",
    "air_temp_k": 300.5,
    "process_temp_k": 310.9,
    "rpm": 1500,
    "torque_nm": 55.0,
    "tool_wear_min": 215,
}

test_df = pd.DataFrame([test_sample])
test_X = make_features(test_df)[feats]
test_X_scaled = scaler.transform(test_X)

proba = float(loaded_model.predict_proba(test_X_scaled)[0, 1])
pred = loaded_model.predict(test_X_scaled)[0]

print(f"Model: {model_uri} (version {mv_loaded.version})")
print(f"\nInput features:")
print(test_X.T)
print(f"\nFailure probability: {proba:.3f}")
print(f"Prediction: {'FAILURE' if pred == 1 else 'NORMAL'}")
print(f"\n✅ Inference test complete")

## 📝 Summary

**Online Learning dengan SGDClassifier**:

✅ **Keuntungan**:
- Model bisa di-update secara incremental tanpa retrain dari nol
- Hemat komputasi dan memori
- Adaptif terhadap perubahan data (concept drift)
- Cocok untuk data streaming

⚠️ **Keterbatasan**:
- Performa tergantung pada urutan data
- Perlu monitoring drift secara berkala
- Tidak semua algoritma support `partial_fit`

**Concept Drift Detection**:
- KS test untuk deteksi perubahan distribusi fitur
- Monitoring metrics (F1, precision, recall) per batch
- Jika drift terdeteksi → update model dengan data baru

**Next Steps**:
- Implementasi di production dengan data streaming real-time
- Gunakan library `river` untuk online learning yang lebih advanced
- Setup automated drift detection dan model retraining pipeline